In [7]:
import pandas as pd
import numpy as np

# Q1

## a)

In [4]:
P_H=0.60
P_D=0.40

P_A_given_H=0.30
P_A_given_D=0.20

P_A=(P_A_given_H*P_H)+(P_A_given_D*P_D)

P_H_given_A=(P_A_given_H*P_H)/P_A

print("Probability that student is a hosteler given A grade:", P_H_given_A)

Probability that student is a hosteler given A grade: 0.6923076923076923


## b)

In [5]:
P_Disease=0.01
P_No_Disease=1-P_Disease

P_Positive_given_Disease=0.99
P_Positive_given_No_Disease=0.02

P_Positive=(P_Positive_given_Disease*P_Disease)+(P_Positive_given_No_Disease*P_No_Disease)

P_Disease_given_Positive=(P_Positive_given_Disease*P_Disease)/P_Positive

print("Probability of having disease given a positive test:",P_Disease_given_Positive)

Probability of having disease given a positive test: 0.3333333333333333


# Q2

In [12]:
df=pd.read_csv('buyers.csv')
df.head(3)

,age,income,student,credit_rating,buys_computer
0,<=30,high,no,fair,no
1,<=30,high,no,excellent,no
2,31...40,high,no,fair,yes


In [14]:
data=df.to_dict("records")

target="buys_computer"

features=[]

for column in df.columns:
    if column != target:
        features.append(column)


classes=[]

for row in data:
    value=row[target]

    if value not in classes:
        classes.append(value)


total_samples=0

for row in data:
    total_samples+=1


class_counts={}

for cls in classes:
    count=0

    for row in data:
        if row[target] == cls:
            count+=1

    class_counts[cls]=count


priors={}

for cls in classes:
    priors[cls]=class_counts[cls]/total_samples


feature_values={}

for feature in features:
    values=[]

    for row in data:
        value=row[feature]

        if value not in values:
            values.append(value)

    feature_values[feature]=values


def predict(test_sample):

    probabilities={}

    for cls in classes:

        probability=priors[cls]

        for feature in features:

            count=0

            for row in data:

                if row[target] == cls and row[feature] == test_sample[feature]:
                    count+=1

            number_of_values=0

            for value in feature_values[feature]:
                number_of_values+=1

            conditional_probability=(
                count+1
            )/(
                class_counts[cls]+number_of_values
            )

            probability*=conditional_probability

        probabilities[cls]=probability


    predicted_class=classes[0]
    max_probability=probabilities[classes[0]]

    for cls in classes:

        if probabilities[cls] > max_probability:
            max_probability=probabilities[cls]
            predicted_class=cls


    return predicted_class, probabilities


test_sample={
    "age":"<=30",
    "income":"medium",
    "student":"yes",
    "credit_rating":"fair"
}


prediction, probabilities=predict(test_sample)


print("Class counts:")
print(class_counts)

print("\nPrior probabilities:")
print(priors)

print("\nTest sample:")
print(test_sample)

print("\nProbabilities:")

for cls in classes:
    print(cls, "=", probabilities[cls])

print("\nPrediction:", prediction)

Class counts:
{'no': 5, 'yes': 9}

Prior probabilities:
{'no': 0.35714285714285715, 'yes': 0.6428571428571429}

Test sample:
{'age': '<=30', 'income': 'medium', 'student': 'yes', 'credit_rating': 'fair'}

Probabilities:
no = 0.008199708454810493
yes = 0.027117768595041326

Prediction: yes


# Q3

In [15]:
import pandas as pd


df=pd.read_csv("text.csv")

for i in range(len(df)):
    df.loc[i,"Text"]=df.loc[i,"Text"].lower()

classes=[]

for i in range(len(df)):
    tag=df.loc[i,"Tag"]

    if tag not in classes:
        classes.append(tag)


class_count={}

for cls in classes:
    count=0

    for i in range(len(df)):
        if df.loc[i,"Tag"] == cls:
            count+=1

    class_count[cls]=count


total_documents=len(df)


prior={}

for cls in classes:
    prior[cls]=class_count[cls]/total_documents


vocabulary=[]

for i in range(len(df)):

    words=df.loc[i,"Text"].split()

    for word in words:

        if word not in vocabulary:
            vocabulary.append(word)


vocabulary_size=len(vocabulary)


word_count={}

total_words_in_class={}

for cls in classes:

    word_count[cls]={}
    total_words=0

    for word in vocabulary:
        word_count[cls][word]=0

    for i in range(len(df)):

        if df.loc[i,"Tag"] == cls:

            words=df.loc[i,"Text"].split()

            for word in words:
                word_count[cls][word]+=1
                total_words+=1

    total_words_in_class[cls]=total_words


def predict(sentence):

    sentence=sentence.lower()

    words=sentence.split()

    probability={}

    for cls in classes:

        prob=prior[cls]

        for word in words:

            if word in vocabulary:
                count=word_count[cls][word]
            else:
                count=0

            conditional_probability=(
                count+1
            )/(
                total_words_in_class[cls]+vocabulary_size
            )

            prob*=conditional_probability

        probability[cls]=prob


    predicted_class=classes[0]
    max_probability=probability[classes[0]]

    for cls in classes:

        if probability[cls] > max_probability:
            max_probability=probability[cls]
            predicted_class=cls


    return predicted_class, probability


sentence="A very close game"

prediction, probabilities=predict(sentence)


print("Sentence:", sentence)

print("\nPrior Probabilities:")

for cls in classes:
    print(cls, "=", prior[cls])


print("\nVocabulary:")
print(vocabulary)


print("\nFinal Probabilities:")

for cls in classes:
    print(cls, "=", probabilities[cls])


print("\nPredicted Tag:", prediction)

Sentence: A very close game

Prior Probabilities:
Sports = 0.6
Not sports = 0.4

Vocabulary:
['a', 'great', 'game', 'the', 'election', 'was', 'over', 'very', 'clean', 'match', 'but', 'forgettable', 'it', 'close']

Final Probabilities:
Sports = 2.7647999999999997e-05
Not sports = 5.7175324559303314e-06

Predicted Tag: Sports


# Additional Q3

In [16]:
import pandas as pd
import re


def preprocess(text):
    text=text.lower()

    # Remove punctuation
    text=re.sub(r"[^a-zA-Z\s]", "", text)

    words=text.split()

    return words


def train_naive_bayes(train_df):

    classes=[]

    for i in range(len(train_df)):
        label=train_df.iloc[i]["Label"]

        if label not in classes:
            classes.append(label)


    class_count={}

    for cls in classes:
        count=0

        for i in range(len(train_df)):
            if train_df.iloc[i]["Label"]==cls:
                count+=1

        class_count[cls]=count


    prior={}

    total_documents=len(train_df)

    for cls in classes:
        prior[cls]=class_count[cls]/total_documents


    vocabulary=[]

    for i in range(len(train_df)):

        words=preprocess(train_df.iloc[i]["Text"])

        for word in words:

            if word not in vocabulary:
                vocabulary.append(word)


    word_count={}

    total_words_in_class={}


    for cls in classes:

        word_count[cls]={}

        for word in vocabulary:
            word_count[cls][word]=0


        total_words=0


        for i in range(len(train_df)):

            if train_df.iloc[i]["Label"]==cls:

                words=preprocess(train_df.iloc[i]["Text"])

                for word in words:
                    word_count[cls][word]+=1
                    total_words+=1


        total_words_in_class[cls]=total_words


    return (
        classes,
        class_count,
        prior,
        vocabulary,
        word_count,
        total_words_in_class
    )


def predict(
    text,
    classes,
    class_count,
    prior,
    vocabulary,
    word_count,
    total_words_in_class
):

    words=preprocess(text)

    probabilities={}


    for cls in classes:

        probability=prior[cls]


        for word in words:

            if word in vocabulary:
                count=word_count[cls][word]

            else:
                count=0


            conditional_probability=(
                count+1
            )/(
                total_words_in_class[cls]+len(vocabulary)
            )


            probability*=conditional_probability


        probabilities[cls]=probability


    predicted_class=classes[0]
    max_probability=probabilities[classes[0]]


    for cls in classes:

        if probabilities[cls]>max_probability:

            max_probability=probabilities[cls]
            predicted_class=cls


    return predicted_class


def calculate_metrics(actual,predicted):

    correct=0

    for i in range(len(actual)):
        if actual[i]==predicted[i]:
            correct+=1


    accuracy=correct/len(actual)


    TP=0
    TN=0
    FP=0
    FN=0


    for i in range(len(actual)):

        if actual[i]=="pos" and predicted[i]=="pos":
            TP+=1

        elif actual[i]=="neg" and predicted[i]=="neg":
            TN+=1

        elif actual[i]=="neg" and predicted[i]=="pos":
            FP+=1

        elif actual[i]=="pos" and predicted[i]=="neg":
            FN+=1


    if TP+FP==0:
        precision=0
    else:
        precision=TP/(TP+FP)


    if TP+FN==0:
        recall=0
    else:
        recall=TP/(TP+FN)


    return accuracy,precision,recall,TP,TN,FP,FN


df=pd.read_csv("sentiment.csv")


train_df=df.iloc[:14].reset_index(drop=True)

test_df=df.iloc[14:].reset_index(drop=True)



(
    classes,
    class_count,
    prior,
    vocabulary,
    word_count,
    total_words_in_class
)=train_naive_bayes(train_df)


actual=[]
predicted=[]


for i in range(len(test_df)):

    text=test_df.iloc[i]["Text"]
    true_label=test_df.iloc[i]["Label"]


    prediction=predict(
        text,
        classes,
        class_count,
        prior,
        vocabulary,
        word_count,
        total_words_in_class
    )


    actual.append(true_label)
    predicted.append(prediction)


    print("Text:",text)
    print("Actual:",true_label)
    print("Predicted:",prediction)
    print()



accuracy,precision,recall,TP,TN,FP,FN=calculate_metrics(
    actual,
    predicted
)


print("Confusion Matrix Values")
print("TP =",TP)
print("TN =",TN)
print("FP =",FP)
print("FN =",FN)


print("\nAccuracy =",accuracy)
print("Precision =",precision)
print("Recall =",recall)


print("\nAccuracy Percentage =",accuracy*100,"%")
print("Precision Percentage =",precision*100,"%")
print("Recall Percentage =",recall*100,"%")

Text: What a great holiday
Actual: pos
Predicted: pos

Text: That is a bad locality to stay
Actual: neg
Predicted: pos

Text: We will have good fun tomorrow
Actual: pos
Predicted: pos

Text: I went to my enemy's house today
Actual: neg
Predicted: pos

Confusion Matrix Values
TP = 2
TN = 0
FP = 2
FN = 0

Accuracy = 0.5
Precision = 0.5
Recall = 1.0

Accuracy Percentage = 50.0 %
Precision Percentage = 50.0 %
Recall Percentage = 100.0 %
